# 02 – Retrieve, Rerank & Generate (Grounded RAG)

This notebook demonstrates a full **Retrieval-Augmented Generation (RAG)** pipeline using:
- **FAISS** – vector similarity search for initial retrieval
- **Cohere Rerank** – cross-encoder reranking to improve relevance
- **Cohere Command** – grounded generation with cited sources

Interim results are printed at each stage so you can inspect how the pipeline works.

> **Prerequisites**  
> - Run `01_embed_documents.ipynb` first to create the `./faiss_index/` folder.  
> - Set `COHERE_API_KEY` in your environment or `.env` file.

## 1. Imports and configuration

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_cohere import ChatCohere, CohereEmbeddings, CohereRerank
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Load environment variables from .env if present
load_dotenv()

# --- Configuration -----------------------------------------------------------
FAISS_INDEX       = Path("./faiss_index")    # path to the saved FAISS index

# Cohere model names (most current as of 2025)
EMBED_MODEL       = "embed-english-v3.0"
RERANK_MODEL      = "rerank-english-v3.0"
GENERATION_MODEL  = "command-r-plus-08-2024"

# Retrieval parameters
RETRIEVAL_K       = 10   # number of candidates from vector search
RERANK_TOP_N      = 3    # how many to keep after reranking
# -----------------------------------------------------------------------------

cohere_api_key = os.environ.get("COHERE_API_KEY")
if not cohere_api_key:
    raise EnvironmentError(
        "COHERE_API_KEY is not set. "
        "Add it to your environment or create a .env file with COHERE_API_KEY=<your-key>."
    )

print("✅ Configuration OK")

## 2. Load the FAISS vector store

In [ ]:
if not FAISS_INDEX.exists():
    raise FileNotFoundError(
        f"FAISS index not found at '{FAISS_INDEX}'. "
        "Please run '01_embed_documents.ipynb' first."
    )

embeddings = CohereEmbeddings(
    model=EMBED_MODEL,
    cohere_api_key=cohere_api_key,
)

vector_store = FAISS.load_local(
    str(FAISS_INDEX),
    embeddings,
    allow_dangerous_deserialization=True,
)

print(f"📦 FAISS index loaded from '{FAISS_INDEX}'")

## 3. Define the query

In [ ]:
# ✏️  Change this to any question you want to ask about your documents
QUERY = "What are the main topics covered in the documents?"

print(f"🤔 Query: {QUERY}")

## 4. Step 1 – Vector similarity retrieval

We perform a **semantic similarity search** against the FAISS index to get the top-K candidate chunks.

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": RETRIEVAL_K})

retrieved_docs = retriever.invoke(QUERY)

print(f"🔍 Retrieved {len(retrieved_docs)} candidate(s) from FAISS\n")
print("=" * 70)
for i, doc in enumerate(retrieved_docs, 1):
    source = doc.metadata.get("source", "unknown")
    page   = doc.metadata.get("page", "?")
    print(f"\n[{i}] Source: {source} | Page: {page}")
    print("-" * 70)
    print(doc.page_content[:400])
print("\n" + "=" * 70)

## 5. Step 2 – Cohere Rerank

We feed all retrieved candidates through **Cohere Rerank** (a cross-encoder model) to re-score them by actual relevance to the query and keep only the top-N.

In [ ]:
compressor = CohereRerank(
    model=RERANK_MODEL,
    cohere_api_key=cohere_api_key,
    top_n=RERANK_TOP_N,
)

reranked_docs = compressor.compress_documents(retrieved_docs, QUERY)

print(f"🏆 After reranking – top {len(reranked_docs)} document(s)\n")
print("=" * 70)
for i, doc in enumerate(reranked_docs, 1):
    source        = doc.metadata.get("source", "unknown")
    page          = doc.metadata.get("page", "?")
    relevance     = doc.metadata.get("relevance_score", "n/a")
    print(f"\n[{i}] Source: {source} | Page: {page} | Relevance score: {relevance:.4f}" if isinstance(relevance, float) else f"\n[{i}] Source: {source} | Page: {page} | Relevance score: {relevance}")
    print("-" * 70)
    print(doc.page_content[:400])
print("\n" + "=" * 70)

## 6. Step 3 – Grounded generation with Cohere Command

We build a prompt that includes the reranked context chunks and asks **Cohere Command** to answer the query while grounding its response in the provided documents.

In [ ]:
def format_docs(docs: list[Document]) -> str:
    """Format a list of documents into a numbered context block."""
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown")
        page   = doc.metadata.get("page", "?")
        formatted.append(
            f"[Document {i}]\n"
            f"Source: {source}, Page: {page}\n"
            f"{doc.page_content}"
        )
    return "\n\n".join(formatted)


RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant that answers questions strictly based on the "
     "provided context documents. "
     "If the answer cannot be found in the documents, say so clearly. "
     "Always cite which document(s) support your answer."),
    ("human",
     "Context documents:\n\n{context}\n\n"
     "Question: {question}"),
])

llm = ChatCohere(
    model=GENERATION_MODEL,
    cohere_api_key=cohere_api_key,
    temperature=0.3,
)

print("🤖 LLM configured:", GENERATION_MODEL)

In [ ]:
# Show the context that will be injected into the prompt
context_str = format_docs(reranked_docs)

print("📋 Context passed to the LLM:")
print("=" * 70)
print(context_str)
print("=" * 70)

In [ ]:
# Build and invoke the RAG chain
rag_chain = (
    {"context": RunnablePassthrough(), "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke({"context": context_str, "question": QUERY})

print("💬 Answer:")
print("=" * 70)
print(answer)
print("=" * 70)

## 7. End-to-end pipeline summary

| Stage | Method | Result |
|-------|--------|--------|
| **1 – Retrieve** | FAISS cosine similarity | Top-K candidate chunks |
| **2 – Rerank** | Cohere Rerank (cross-encoder) | Top-N most relevant chunks |
| **3 – Generate** | Cohere Command (grounded) | Cited, context-faithful answer |

In [ ]:
# Convenience helper: run the full pipeline in one call
def rag_query(query: str, k: int = RETRIEVAL_K, top_n: int = RERANK_TOP_N) -> str:
    """Run the full retrieve → rerank → generate pipeline and return the answer."""
    docs       = retriever.invoke(query)
    reranked   = compressor.compress_documents(docs, query)
    context    = format_docs(reranked)
    return rag_chain.invoke({"context": context, "question": query})


# Try it
followup = "Summarise the key findings in one paragraph."
print(f"🤔 Follow-up query: {followup}\n")
print(rag_query(followup))